# 02 — Transcripts

Two paths controlled by the switch below.

1. **Import path (default).** Walk the five course-content folders in `raw-data/` and read the existing `*.transcript.csv` files (segment-level: `start, end, text`).
2. **Whisper path.** For any lecture without a pre-computed transcript (or all lectures, if the switch is flipped), call OpenAI's `whisper-1` via the API with `language="sk"` + a domain-keyword prompt. Output is `verbose_json`.

**Output:** `artefacts/transcripts.parquet` with columns `(course_id, lecture_id, segment_idx, t_start, t_end, text)`.

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = Path('raw-data')
OUT = Path('artefacts')
OUT.mkdir(exist_ok=True)

USE_EXISTING_TRANSCRIPTS = True

WHISPER_PROMPT = (
    'BBC micro:bit, Microsoft MakeCode, MicroPython, '
    'pin, neopixel, akcelerometer, kalibrácia'
)
WHISPER_LANGUAGE = 'sk'

COURSE_DIR_TO_ID = {
    'course-microbit-makecode-zakladny-content':   1,
    'course-microbit-makecode-pokrocily-content':  5,
    'course-inovacne_vzdelavanie-content':        30,
    'course-inovacne_vzdelavanie2-content':       32,
    'course-pbl-akademia-content':                42,
}

## Path 1 — Import existing transcripts

In [2]:
def import_transcripts() -> pd.DataFrame:
    rows = []
    for course_dir, course_id in COURSE_DIR_TO_ID.items():
        course_path = RAW / course_dir
        if not course_path.exists():
            print(f'  skip (missing): {course_path}')
            continue
        for csv_path in sorted(course_path.rglob('*.transcript.csv')):
            lecture_id = csv_path.name.replace('.transcript.csv', '')
            chapter_dir = csv_path.parent.name
            df = pd.read_csv(csv_path)
            df['course_id']   = course_id
            df['chapter']     = chapter_dir
            df['lecture_id']  = lecture_id
            df['segment_idx'] = range(len(df))
            rows.append(df)
    out = pd.concat(rows, ignore_index=True)
    return out.rename(columns={'start': 't_start', 'end': 't_end'})[
        ['course_id', 'chapter', 'lecture_id', 'segment_idx', 't_start', 't_end', 'text']
    ]

if USE_EXISTING_TRANSCRIPTS:
    transcripts = import_transcripts()
    print(f'imported {len(transcripts):,} segment rows '
          f'across {transcripts.groupby(["course_id","lecture_id"]).ngroups} lectures '
          f'in {transcripts.course_id.nunique()} courses')
    print(f'\nper-course lecture counts:')
    print(transcripts.groupby('course_id').lecture_id.nunique())

imported 14,052 segment rows across 159 lectures in 5 courses

per-course lecture counts:
course_id
1     31
5     36
30    29
32    35
42    28
Name: lecture_id, dtype: int64


## Path 2 — Whisper transcription (fallback)

Used only when `USE_EXISTING_TRANSCRIPTS = False`.

In [3]:
# Stub. Activate by setting USE_EXISTING_TRANSCRIPTS = False above and uncommenting.
# from openai import OpenAI
# client = OpenAI()
#
# def transcribe_video(mp4_path):
#     with open(mp4_path, 'rb') as f:
#         resp = client.audio.transcriptions.create(
#             model='whisper-1',
#             file=f,
#             language=WHISPER_LANGUAGE,
#             prompt=WHISPER_PROMPT,
#             response_format='verbose_json',
#             timestamp_granularities=['segment', 'word'],
#         )
#     return resp

## Persist

In [4]:
transcripts.to_parquet(OUT / 'transcripts.parquet', index=False)
print(f'wrote {OUT / "transcripts.parquet"} ({(OUT / "transcripts.parquet").stat().st_size/1024:.0f} KB)')

transcripts.head()

wrote artefacts/transcripts.parquet (381 KB)


,course_id,chapter,lecture_id,segment_idx,t_start,t_end,text
0,1,01-kapitola-1,01-uvod-do-kurzu,0,0.00,3.48,"Ahojte, som Marek Mansell a spoločne s Nikou K..."
1,1,01-kapitola-1,01-uvod-do-kurzu,1,3.48,6.24,vás budeme sprevádzať týmto micro:bit videokur...
2,1,01-kapitola-1,01-uvod-do-kurzu,2,6.24,9.28,"Celý kurz je rozčlenený do viacerých kapitol,"
3,1,01-kapitola-1,01-uvod-do-kurzu,3,9.28,11.68,ktoré obsahujú jednotlivé videolekcie.
4,1,01-kapitola-1,01-uvod-do-kurzu,4,11.68,16.08,"V tomto prvom videu si predstavíme, ako bude k..."
